# Autonomous Multi-Agent Research Analyst
### Built with LangChain Deep Agents + Groq (Llama 3.3 70B)

---

## Architecture

```
topic
  │
  ▼
[MEMORY CHECK]   ← what do we already know?
  │
  ▼
[HITL GATE]      ← stream research plan → user approves / redirects / cancels
  │ approved
  ▼
Web Searcher ──── raw sources ────┬──────────────────────┐
                                  ▼                      ▼
                        Insight Extractor          Fact Checker
                        (uses summarizer skill)    (uses extract_claims)
                                  │                      │
                                  └──────────┬───────────┘
                                             ▼
                              Orchestrator  (streamed live)
                                             │
                              ┌──────────────┴──────────────┐
                              ▼                             ▼
                       memory/store.json          sessions/<id>.json
                       (compressed knowledge)     (full audit log)
```

## Features covered

| # | Concept | Where |
|---|---|---|
| 1 | Orchestrator + parallel subagents | `orchestrator.py` + `subagents/` |
| 2 | Human-in-the-Loop | `hitl_gate()` in orchestrator |
| 3 | Streaming & observability | `streaming.py` |
| 4 | Long-term memory | `memory/memory.py` |
| 5 | Filesystem backend | `backend/filesystem.py` |
| 6 | Reusable skills | `skills/citation_formatter.py`, `skills/summarizer.py` |

---
## 1 · Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # make research_analyst/ the root

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to Deep Agent/.env"
print("✓ API key loaded")

---
## 2 · Skills — reusable tools shared across agents

Skills are plain Python functions defined once in `skills/` and imported by any agent that needs them.  
This avoids duplicating logic across subagents.

In [ ]:
from skills.citation_formatter import format_citation
from skills.summarizer import summarize

# Citation formatter — used by Web Searcher
print(format_citation(
    title="Quantum Computing Overview",
    url="https://en.wikipedia.org/wiki/Quantum_computing",
    author="Wikipedia",
    year="2025",
    style="apa",
))
print()

# Summarizer — used by Insight Extractor
long_text = (
    "Quantum computing is a type of computation that harnesses quantum phenomena. "
    "Unlike classical computers that use bits, quantum computers use qubits. "
    "A landmark 2024 study showed a 40% improvement over prior benchmarks. "
    "Key players include IBM, Google, and IonQ. "
    "Open challenges include error correction, decoherence, and scalability. "
    "The market is projected to reach $500B by 2030. "
    "Researchers at MIT published a breakthrough paper on topological qubits."
)
print(summarize(long_text, max_sentences=3, focus="breakthrough"))

---
## 3 · Subagents — each one is a focused `create_deep_agent`

Every subagent has:
- a narrow **responsibility**
- its own **tools** (including shared skills)
- its own **system prompt**

In [ ]:
from subagents import web_searcher

# Web Searcher: calls web_search + format_citation (skill)
raw = web_searcher.run("Quantum Computing")
print(raw)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from subagents import insight_extractor, fact_checker

# Insight Extractor + Fact Checker run in PARALLEL on the same raw sources
t0 = time.time()
results = {}

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {
        pool.submit(insight_extractor.run, raw): "insight_extractor",
        pool.submit(fact_checker.run, raw):       "fact_checker",
    }
    for f in as_completed(futures):
        name = futures[f]
        results[name] = f.result()
        print(f"  ✓ {name} done")

print(f"\nBoth finished in {time.time()-t0:.1f}s (parallel)")

In [ ]:
print("=== INSIGHTS ===")
print(results["insight_extractor"])
print("\n=== FACT CHECK ===")
print(results["fact_checker"])

---
## 4 · Streaming — real-time token output

`stream_agent()` is a drop-in for `.invoke()` that prints tokens as they arrive  
and returns the full text when done.

In [ ]:
from deepagents import create_deep_agent
from streaming import stream_agent

orchestrator = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[],
    system_prompt=(
        "You are the lead research orchestrator. Synthesise the subagent outputs "
        "into a polished report with sections: Executive Summary, Key Findings, "
        "Core Insights, Reliability Assessment, Next Steps."
    ),
)

synthesis_prompt = f"""
Research topic: Quantum Computing

=== WEB SEARCHER OUTPUT ===
{raw}

=== INSIGHT EXTRACTOR OUTPUT ===
{results['insight_extractor']}

=== FACT CHECKER OUTPUT ===
{results['fact_checker']}
""".strip()

report = stream_agent(orchestrator, synthesis_prompt, label="Final Report")

---
## 5 · Long-Term Memory — knowledge that persists across runs

In [ ]:
from memory import memory

# Save the report to memory
memory.remember("Quantum Computing", report)
print("Saved to memory.")

In [ ]:
# Exact recall
entry = memory.recall("Quantum Computing")
print(f"Topic          : {entry['topic']}")
print(f"Research count : {entry['research_count']}")
print(f"Last updated   : {entry['last_updated']}")
print(f"Keywords       : {', '.join(entry['keywords'][:8])}")

In [ ]:
# Fuzzy recall — finds related topics by keyword overlap
related = memory.recall_related("Quantum Hardware")
for r in related:
    print(f"Related: '{r['topic']}'  (researched {r['research_count']}x)")

---
## 6 · Filesystem Backend — full audit log per session

In [ ]:
from backend import filesystem

session_id = filesystem.save_session(
    topic="Quantum Computing",
    subagent_outputs={
        "web_searcher":      raw,
        "insight_extractor": results["insight_extractor"],
        "fact_checker":      results["fact_checker"],
    },
    report=report,
)
print(f"Session ID: {session_id}")

In [ ]:
# List all saved sessions
sessions = filesystem.list_sessions()
for s in sessions:
    print(f"[{s['timestamp']}]  {s['topic']}  →  {s['id']}")

In [ ]:
import json

# Load and inspect a session — every subagent output is preserved
session = filesystem.load_session(session_id)
print(json.dumps({
    "id":        session["id"],
    "topic":     session["topic"],
    "timestamp": session["timestamp"],
    "subagents": {k: v[:60]+"..." for k, v in session["subagents"].items()},
}, indent=2))

---
## 7 · Full pipeline — everything in one call

This runs the complete flow: memory check → HITL gate → parallel subagents → streamed report → memory update → session save.

> **Note:** The HITL gate will pause and ask for your input (`y / n / r`). Type `y` to proceed.

In [ ]:
import orchestrator
orchestrator.run("Artificial Intelligence Safety")